# Sesli kitap: seslendirme ve test

Çalışma zamanı: **T4 GPU**. Tümünü çalıştırın. 2. hücre `is-<konu>.zip` ister; 4. hücre `cikti-<konu>.zip`'i base64 olarak yazar. Yeni tur için yalnız 2-4. hücreler yeniden çalıştırılır (modeller önbellekte kalır).

Yalnız açık kaynak model çalışır (pocket-tts-tr, Whisper, wav2vec2, UTMOS). Kişisel veri yüklenmez.

In [ ]:
# 1. GPU denetimi
!nvidia-smi -L || echo 'GPU yok: Runtime > Change runtime type > T4 GPU'

In [ ]:
# 2. Drive'ı bağla (çıktı orada kalır, yarım iş kaldığı yerden sürer) ve iş dosyasını yükle (is-<konu>.zip)
import pathlib, shutil, zipfile
from google.colab import drive, files
drive.mount("/content/drive")

yuklenen = files.upload()
ad = next(n for n in yuklenen if n.startswith("is-") and n.endswith(".zip"))
IS = pathlib.Path("/content/is")
shutil.rmtree(IS, ignore_errors=True)
zipfile.ZipFile(ad).extractall(IS)
pathlib.Path(ad).unlink()
print("İş açıldı:", ad, sorted(p.name for p in IS.iterdir()))

In [ ]:
# 3. Seslendir ve ölç
!bash /content/is/calistir.sh

In [ ]:
# 4. Çıktıyı hazırla: base64 parçalara bölünür (tek seferde Colab çıktı sınırını aşıyordu).
# Ajan 5. hücreyi N kez çalıştırır; her çalıştırma P<k>START...P<k>END yazar. Zip Drive'da da durur.
import base64, glob, math
cikti = max(glob.glob("/content/is/cikti-*.zip") + glob.glob("/content/drive/MyDrive/sesli_kitap/cikti-*.zip"))
B = base64.b64encode(open(cikti, "rb").read()).decode()
N = math.ceil(len(B) / 6_000_000); L = math.ceil(len(B) / N); k = -1
print("Çıktı:", cikti, len(B), "N =", N)

In [ ]:
# 5. Sıradaki parçayı yaz (N kez çalıştırın)
k += 1
print(f"P{k}START" + B[k * L:(k + 1) * L] + f"P{k}END" if k < N else "BİTTİ")